# Day 5: Company Brochure Generator

Given a company name and website, Gemini picks the relevant pages, reads them, and writes a brochure
for prospective customers, investors and recruits.

In [ ]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

load_dotenv(override=True)
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    print("No API key was found. Add GEMINI_API_KEY to your .env file.")
elif not api_key.startswith(("AIz", "AQ.")):
    print("An API key was found, but it doesn't start with AIz or AQ.")
elif api_key.strip() != api_key:
    print("The API key has leading or trailing whitespace.")
else:
    print("API key found and looks good so far!")

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
MODEL = "gemini-3.5-flash"

gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=api_key)

## Step 1: Let Gemini choose the relevant links

One-shot prompting with a JSON example, and `response_format` to force valid JSON.

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""


def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    return user_prompt + "\n".join(fetch_website_links(url))


def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)},
        ],
        response_format={"type": "json_object"},
    )
    links = json.loads(response.choices[0].message.content)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://home.cegsoft.com/")

## Step 2: Gather the content and write the brochure

In [ ]:
def fetch_page_and_all_relevant_links(url):
    result = f"## Landing Page:\n\n{fetch_website_contents(url)}\n## Relevant Links:\n"
    for link in select_relevant_links(url)["links"]:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result


brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    return user_prompt[:5_000]


def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)},
        ],
    )
    return response.choices[0].message.content

In [ ]:
display(Markdown(create_brochure("CEGSoft", "https://home.cegsoft.com/")))

## My extension: translate the brochure to Spanish

A second LLM call chained onto the first one.

In [ ]:
def translate_to_spanish(content):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Translate the following markdown into Spanish, keeping the formatting."},
            {"role": "user", "content": content},
        ],
    )
    return response.choices[0].message.content


display(Markdown(translate_to_spanish(create_brochure("HuggingFace", "https://huggingface.co"))))

## Streaming the brochure

In [ ]:
def stream_brochure(company_name, url):
    stream = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)},
        ],
        stream=True,
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")